In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少包含两个关键的部分:

- **模型**: 是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**: 是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下:

- 定义工具
- 初始化模型
- 初始化Agent,绑定模型和工具

# 自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息:

- 工具名
- 工具的作用
- 工具需要的参数

## 基于tool描述工具

在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：

In [2]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 使用函数名和文档注释描述工具（推荐）

如果不@tool装饰器没有定义工具名和作用描述，此时:

- 工具名: 默认就是函数名
- 工具所需的参数: 默认就是函数的参数列表
- 工具作用的描述: 默认就是函数的文档注释

In [3]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5

In [4]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False ) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 定义Pydantic Model描述参数

如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。

In [5]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal

# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数。"""
    location: str = Field(description="city name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-days forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

工具调用方式与普通函数调用方式一致

In [6]:
square_root.invoke({"x": 467})

21.61018278497431

In [7]:
get_weather.invoke({"location": "杭州", "include_forecast": False})

'Current weather in 杭州: 22 degrees C'

## 测试

In [8]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-chat",
    tools = [square_root, get_weather]
)

In [9]:
for token, _ in agent.stream(
        {"messages": [HumanMessage(content="杭州接下来几天天气如何？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

I'll look up the weather for Hangzhou including the forecast.Current weather in Hangzhou: 22 degrees C
Next 5 days: Sunny杭州接下来几天的天气情况如下：

**当前天气**：22°C

**未来 5 天**：以晴天为主 ☀️

整体来看天气不错，气温适宜，适合外出活动。不过预报只给出了「晴天」这一概括信息，没有具体的气温范围和是否有降雨的细节——如果你需要更精确的逐日温度或降水信息，建议再查看当地气象台的详细预报。

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少？")]}
)
for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少？
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_00_jkBtOfyAZqWLwAr6YzMW2734)
 Call ID: call_00_jkBtOfyAZqWLwAr6YzMW2734
  Args:
    x: 467
  square_root (call_01_UIs8PftaCfVikodE5Dpl9591)
 Call ID: call_01_UIs8PftaCfVikodE5Dpl9591
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算结果如下：

- **√467 ≈ 21.6102**（467 不是完全平方数）
- **√529 = 23**（因为 23² = 529，正好是完全平方数）
None


# 预定义Tool

LangChain中提供了很多预定义的Tool,方便我们使用。例如:

- tavily: 就是一个用来做web搜索的工具

## 基本用法

它的试用步骤是这样的:

- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖: uv add langchain-tavily

In [11]:
# 使用tavily作为web搜索工具
import importlib.util
import os
import sys

# 自检 1：包在当前 kernel 里能不能找到（找不到通常是 kernel 没重启 / 解释器不是 .venv）
if importlib.util.find_spec("langchain_tavily") is None:
    raise RuntimeError(
        "langchain_tavily 在当前 kernel 里找不到，但它确实已经装在 .venv 里。\n"
        "当前 kernel 解释器: " + sys.executable + "\n"
        "处理办法: 点右上角「IDE 托管服务器」重启 kernel，确认解释器是 .venv [3.14.6]，再重跑本单元。"
    )

# 自检 2：环境变量有没有真正灌进 kernel
if not os.getenv("TAVILY_API_KEY"):
    raise RuntimeError("TAVILY_API_KEY 没读到：请先跑第 1 单元的 load_dotenv()")

from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general", # general, news ,finance
)
print("TavilySearch 就绪 | key 长度 =", len(os.environ["TAVILY_API_KEY"]))


TavilySearch 就绪 | key 长度 = 57


In [12]:
search_tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.sina.cn/news/detail/5247814261015270.html',
   'title': '#周深 蒸蚌#“蒸蚌”=“真棒”的谐音梗，这次出圈就是“萝卜纸巾猫”二创的番外彩蛋：1. 梗怎么来的 ◦ 起源是抖音博主@超级无敌大开门训练三花猫分辨“萝卜/纸巾”，猫咪靠“察言观色”蒙答案，主人用米老鼠式高音喊“真棒”。 ◦ 视频被搬运到微博后，评论区把“真棒”空耳成“蒸蚌”，配上猫咪 \u200b_新浪新闻',
   'content': '发布于 广西\n\n首页\n\n导航\n\n反馈 [...] 汽车 本地生活 旅游 数码 娱乐 影视\n\n财娱乐\n\n25-12-25 18:51 [...] #周深 蒸蚌#  \n“蒸蚌”=“真棒”的谐音梗，这次出圈就是“萝卜纸巾猫”二创的番外彩蛋：  \n1. 梗怎么来的  \n ◦ 起源是抖音博主@超级无敌大开门训练三花猫分辨“萝卜/纸巾”，猫咪靠“察言观色”蒙答案，主人用米老鼠式高音喊“真棒”。  \n ◦ 视频被搬运到微博后，评论区把“真棒”空耳成“蒸蚌”，配上猫咪一脸懵的截图，形成魔性谐音。  \n2. 周深怎么“接招”  \n ◦ 12-23某品牌直播里，主持人拿出“萝卜纸巾猫”同款色卡让周深秒答，他0.01秒答对后顺势喊了声“蒸蚌”，现场瞬间爆笑。  \n ◦ 工作室立刻跟进，抽签名“蒸蚌”小卡，把梗做成官方周边，完成“明星二创→粉丝三创”的闭环。  \n3. 为什么火  \n ◦ 低门槛+强反差：任何人都能对宠物/朋友喊一句“蒸蚌”，瞬间get“猫咪蒙题”的喜感。  \n ◦ 8G网速人设：周深一向“网上冲浪第一名”，观众爱看他“活学活用”热梗，形成“猫传人→明星传粉”的连锁反应。  \n4. 小提醒  \n ◦ 对宠物模仿时记得“只拍照别真考”，别让猫狗过度训练；  \n ◦ 目前“蒸蚌”暂无额外含义，就是“真棒”的可爱读法，放心用。',
   'score': 0.86632514,
   'raw_content': None,
   'id': 'ef6

In [13]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model = "deepseek-chat",
    tools = [search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [14]:
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]}
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================

我来帮你查一下“蒸蚌”这个梗的来源。
Tool Calls:
  tavily_search (call_00_bhff8XQ8QctKcJRVRjZC2998)
 Call ID: call_00_bhff8XQ8QctKcJRVRjZC2998
  Args:
    query: 蒸蚌 是什么梗 出处
    search_depth: advanced
  tavily_search (call_01_O1LC9ktBHwLfAdpIM1OC5290)
 Call ID: call_01_O1LC9ktBHwLfAdpIM1OC5290
  Args:
    query: "蒸蚌" 网络流行语 意思
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 是什么梗 出处", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.adquan.com/article/359318", "title": "萝卜纸巾分不清的“蒸蚌小猫”，拿下首个全球代言！ | 广告门", "content": "“动物营销”的信任优势：\n\n相比于人类明星可能出现的“人设塌房”风险，动物IP的形象更加纯粹、稳定且安全。\n\n它们的“人设”几乎不会崩塌，且天生具备亲近感和治愈力，能有效降低消费者对广告的抵触心理，提升品牌好感度。德祐结合公益的举措，更是将这种好感度转化为品牌美誉度。\n\n谐音梗与“梗文化”的胜利：\n\n“蒸蚌”作为“真棒”的谐音，本身就是一个极具记忆点和传播力的符号

## 优化

目前的搜索智能体存在两个问题:

- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路

- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，过于简单的业务，我们建议大家利用tavily自定义工具。

In [15]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体

In [16]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [17]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model = "deepseek-chat",
    tools = [web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [18]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]}
)

# 获取结构化输出
print(response['structured_response'])

answer='「蒸蚌」是「真棒」的谐音梗（空耳），源于2025年底爆火的「萝卜纸巾猫」视频，后来成为全网流行的夸奖/自嘲用语。具体如下：\n\n**1. 梗的起源**\n抖音博主「超级无敌大开门」训练自家三花猫「大开门」分辨「胡萝卜」和「纸巾」。猫咪其实分不清，全靠偷偷瞄主人表情、察言观色来「蒙答案」，选错了立刻改。主人（带湖北口音、用米老鼠式的高音）在猫选对时就会喊一声极具感染力的「真棒！」并给零食奖励。视频被搬到微博等平台后，评论区把「真棒」空耳成「蒸蚌」，再配上猫咪一脸懵的截图，形成魔性谐音梗。\n\n**2. 为什么火**\n- 反差萌：猫咪「似笨实精」「我啥也不会但我会装」的状态，被网友调侃像极了人生考试现场（蒙对答案的侥幸）。\n- 情绪价值：无论对错，主人永远给一句「真棒」，代表一种无条件的鼓励与允许试错的松弛感，精准治愈了害怕犯错、压力大的现代人。\n- 低门槛易二创：任何人都可以对宠物、朋友喊一句「蒸蚌」，衍生出「小猫考研版」「狗子版」「文旅版」「打工人版」等无数二创。\n\n**3. 传播与出圈**\n- 明星接棒：2025年12月某品牌直播中，主持人拿「萝卜纸巾猫」同款色卡考周深，他秒答对后顺势喊出「蒸蚌」，工作室还抽「蒸蚌」签名小卡，把梗做成官方周边（话题「#周深 蒸蚌#」）。\n- 商业联动：这只猫后来被称为「蒸蚌小猫」，甚至拿下品牌全球代言，文旅、品牌纷纷跟风玩梗。\n\n**4. 用法**\n现在「蒸蚌」基本就等于「真棒」的可爱、调侃式读法，用来夸人、夸宠物、自我鼓励或摆烂敷衍都行，本身没有其他额外含义，可以放心使用。\n\n小提醒：如果模仿视频「考」宠物，记得别过度训练，重在拍照娱乐就好。' reference=[Reference(title='#周深 蒸蚌#（新浪新闻）', url='https://www.sina.cn/news/detail/5247814261015270.html'), Reference(title='萝卜纸巾分不清的「蒸蚌小猫」，拿下首个全球代言！（广告门）', url='https://www.adquan.com/article/359318'), Reference(title='萝卜纸巾蒸蚌！全网小猫开始考研！（Dealmoon）', url='https://www.dealmoon.co